In [16]:
# import libraries
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings("ignore")

# set config
results_df = pd.DataFrame(columns=["prompt_template", "prompt_name", "F1_score"])
model_name = "google/flan-t5-small"


In [30]:
# Load the dataset
valid_df = pd.read_csv('data/6/valid.csv')
valid_df.drop(columns=['data_id'], inplace=True)
valid_df.dropna(inplace=True)

In [29]:
# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

def pipeline(df, prompt_template, prompt_name):
    # data cleaning
    df = clean_text(df)

    # Load model
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Set model to eval mode (faster)
    model.eval()

    # Apply in batches
    batch_size = 32
    preds = []

    for i in range(0, len(df), batch_size):
        batch_texts = df["text"].iloc[i:i+batch_size].tolist()
        preds.extend(batch_predict(batch_texts, prompt_template, tokenizer, model))

    df[prompt_name] = [p.strip() for p in preds]

    # evaluation
    df[prompt_name] = df[prompt_name].str.lower()
    df['sentiment'] = df['sentiment'].str.lower()
    df['correct'] = df['sentiment'] == df[prompt_name]

    f1 = f1_score(df['sentiment'], df[prompt_name], average='weighted')
    print(f"F1-Score: {round(100*f1, 1)}%")

    results = {
        "prompt_template": prompt_template,
        "prompt_name": prompt_name,
        "F1_score": f1
    }

    return results, df

In [36]:
# one shot
# f1 = 90.7%
prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_1"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 90.7%


In [35]:
# however
# f1 = 91.4%
prompt = """Classify the sentiment of the text.

Labels: positive , negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_2"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 91.4%


In [34]:
# two shot
# f1 = 89.5%
prompt = """You are a sentiment classifier.
Answer with only one word: positive or negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: negative

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_3"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 89.5%


In [33]:
# question
# f1 = 89.0%
prompt = """
What is the sentiment of this text?

Answer using only one word: positive or negative.

Text: {t}
Answer:"""

prompt_name = "out_label_Prompt_4"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 89.0%


In [ ]:
# explain sentiment
# f1 = 88.7%
prompt = """
Classify sentiment.

Positive = praise, satisfaction, good experience.
Negative = complaint, disappointment, bad experience.

Answer with one word only: positive or negative.

Text: {t}
Answer:"""
prompt_name = "out_label_Prompt_5"
results, df = pipeline(valid_df, prompt, prompt_name)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
valid_df = valid_df.merge(df[prompt_name], left_index=True, right_index=True, how="left")

F1-Score: 88.7%


In [37]:
valid_df

,sentiment,text,out_label_Prompt_5,out_label_Prompt_4,out_label_Prompt_3,out_label_Prompt_2,out_label_Prompt_1
0,positive,These work great and are a better value than m...,positive,positive,positive,positive,positive
1,negative,I replaced the all the suspension rods/shocks ...,negative,negative,negative,negative,negative
2,positive,my old ones couldn't tolerate the heat and bec...,positive,positive,positive,positive,positive
3,negative,This bracket will not fit the under counter wa...,negative,negative,negative,negative,negative
4,positive,These work fairly well and save a lot of money...,positive,positive,positive,positive,positive
...,...,...,...,...,...,...,...
695,negative,Installed this in my frig and a gush of black ...,negative,negative,negative,negative,negative
696,negative,Does not fit my Samsung refrigerator.,negative,negative,negative,negative,negative
697,positive,The video showing how it works was extremely a...,positive,positive,positive,positive,positive
698,positive,"Did have a small issue putting it in, but it w...",positive,positive,positive,positive,positive


In [42]:
results_df

,prompt_template,prompt_name,F1_score
0,"\nClassify sentiment.\n\nPositive = praise, sa...",out_label_Prompt_5,0.886649
1,\nWhat is the sentiment of this text?\n\nAnswe...,out_label_Prompt_4,0.890441
2,You are a sentiment classifier.\nAnswer with o...,out_label_Prompt_3,0.894794
3,Classify the sentiment of the text.\n\nLabels:...,out_label_Prompt_2,0.913923
4,You are a sentiment classifier.\nAnswer with o...,out_label_Prompt_1,0.907411
